# 01 · First contact with vLLM

**Goal:** start vLLM yourself, watch it boot, send it your own questions, and read the raw answers. No benchmark, no scoring yet.
This is the same idea as `ollama serve` + sending it a message, just with vLLM.

**Before you start (right-hand panel → Settings):**
- **Accelerator: GPU T4 x2** (vLLM does not support the P100 option)
- **Internet: On** (to download vLLM and the model)

Kaggle has no terminal window. Cells starting with `!` or `%%bash` run **shell commands**, so they are your terminal.
Run the cells **one at a time** (Shift+Enter) and read the note above each one first.

## Step 1 · Look at your GPUs
**What to look for:** two lines saying *Tesla T4*, each with about *15360MiB* of memory, *0MiB* used, and no processes yet.

In [ ]:
!nvidia-smi

## Step 2 · Install vLLM
This downloads vLLM and its parts. It takes **3-6 minutes** and prints a lot. That's normal.
We pin version **0.18.1** because its PyTorch requirement matches the one Kaggle already has installed.
If the last lines show an **error**, copy them and send them to me.

In [ ]:
!pip install vllm==0.18.1

In [ ]:
!vllm --version

## Step 3 · Start the vLLM server (in the background)
This is the same command you would type in a terminal. Each flag:

| Flag | Meaning |
|---|---|
| `Qwen/Qwen2.5-Coder-1.5B-Instruct` | the model, downloaded from Hugging Face (about 3 GB) |
| `--dtype half` | float16, because the T4 has no bfloat16 |
| `--max-model-len 8192` | context window cap: prompt + answer ≤ 8192 tokens |
| `--gpu-memory-utilization 0.85` | vLLM may use 85 % of the GPU's memory |
| `--tensor-parallel-size 1` | use 1 GPU. The 1.5B model fits on one, and 2 would only add communication overhead |
| `--enforce-eager` | skip "CUDA graph" capture: faster startup, slightly slower generation |
| `--attention-backend TRITON_ATTN` | which code computes attention. The T4 is too old for FlashAttention, and vLLM's next choice, FlashInfer, has to compile itself on Kaggle and fails (`cannot find -lcuda`). Triton works on any GPU |
| `--seed 42` | fixed randomness, so runs are repeatable |
| `--port 8000` | the server listens at http://localhost:8000 |

The server runs in the background, and everything it prints goes into the file `vllm_server.log`.

In [ ]:
import subprocess, shlex

MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
cmd = (f"vllm serve {MODEL} --dtype half --max-model-len 8192 --gpu-memory-utilization 0.85 "
       f"--tensor-parallel-size 1 --enforce-eager --attention-backend TRITON_ATTN --seed 42 --port 8000")
print(cmd)

# if an earlier server from this session is still running, stop it first
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait()

log_file = open("vllm_server.log", "w")
server = subprocess.Popen(shlex.split(cmd), stdout=log_file, stderr=subprocess.STDOUT)
print("Server process started, id:", server.pid)

## Step 4 · Watch it boot (like watching a terminal)
This cell prints the server's log as it grows, until the server answers "healthy" (usually 2-5 minutes the first time,
because it downloads the model).

**Lines worth noticing, in order** (exact wording differs between versions):
1. the settings it uses (model, `dtype=torch.float16`, `max_seq_len=8192`). An `ERROR … FA2 is only supported on devices with
   compute capability >= 8` line is harmless: vLLM is just noting that the T4 is too old for FlashAttention 2
1b. `Using TRITON_ATTN attention backend`, the backend we chose
2. downloading and loading the weights (`Loading weights took … seconds`, `Model loading took … GiB`)
3. ⭐ **the KV cache:** `GPU KV cache size: … tokens` and `Maximum concurrency for 8,192 tokens per request: …x`.
   This is the "drawer of cards" from knowledge base §3-4. **Write these two numbers down.**
4. `Starting vLLM API server … on http://0.0.0.0:8000` and `Application startup complete`

In [ ]:
import time, requests

log_position = 0
def print_new_log_lines():
    global log_position
    with open("vllm_server.log") as f:
        f.seek(log_position)
        new_text = f.read()
        log_position = f.tell()
    if new_text:
        print(new_text, end="")

while True:
    print_new_log_lines()
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("\n✅ SERVER READY")
            break
    except requests.exceptions.ConnectionError:
        pass
    if server.poll() is not None:
        print("\n❌ The server stopped. Send me the last lines of the log above.")
        break
    time.sleep(5)

## Step 5 · Ask the server which model it has
Servers like vLLM (and Ollama, and OpenAI's API) speak the same **"OpenAI-compatible" HTTP language**. This is a plain HTTP request.

In [ ]:
!curl -s http://localhost:8000/v1/models | python -m json.tool

## Step 6 · Your first question, as a raw HTTP request
This is exactly what every chat app sends: a JSON with the model name, the **messages** (with roles), and sampling settings.

**Read the answer JSON:**
- `choices[0].message.content` → the model's answer
- `choices[0].finish_reason` → `"stop"` (finished by itself) or `"length"` (hit `max_tokens`, cut off)
- `usage.prompt_tokens` / `completion_tokens` → tokens in and out. Notice that `prompt_tokens` is **bigger than your question**:
  the chat template adds Qwen's default system message and the `<|im_start|>` markers (knowledge base §5.1).

In [ ]:
%%bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
        "model": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "messages": [{"role": "user", "content": "What is SQL? Answer in two sentences."}],
        "temperature": 0,
        "max_tokens": 200
      }' | python -m json.tool

## Step 7 · The same from Python, so you can play
`ask()` sends one question and prints the parts that matter. **Change the question and run it again.** Try these:
1. `max_tokens=20` → watch `finish_reason` become `"length"` (the answer is cut off)
2. run the same question twice with `temperature=0` → identical answers (greedy decoding)
3. run it twice with `temperature=1.0` → different answers each time
4. ask it in German, or ask it to explain a SQL query

In [ ]:
URL = "http://localhost:8000/v1/chat/completions"

def ask(question, temperature=0, max_tokens=500):
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": question}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    t0 = time.time()
    response = requests.post(URL, json=payload).json()
    seconds = time.time() - t0
    choice, usage = response["choices"][0], response["usage"]
    print(choice["message"]["content"])
    print("\n---")
    print(f"finish_reason: {choice['finish_reason']} | prompt tokens: {usage['prompt_tokens']} | "
          f"answer tokens: {usage['completion_tokens']} | {seconds:.1f} s | "
          f"{usage['completion_tokens'] / seconds:.0f} tokens/s")
    return response

ask("Write a SQLite query that returns the 5 tallest superheroes "
    "from a table superhero(id, superhero_name, height_cm).")

## Step 8 · Under the hood: what the model *really* receives
vLLM can show you the chat template at work. `/tokenize` turns your messages into **token IDs**, and `/detokenize` turns those
IDs back into text, **including the special markers** the model sees (knowledge base §5.1).

In [ ]:
messages = [{"role": "user", "content": "What is SQL?"}]

tok = requests.post("http://localhost:8000/tokenize",
                    json={"model": MODEL, "messages": messages, "add_generation_prompt": True}).json()
print("number of tokens:", tok.get("count"))
print("token IDs:", tok.get("tokens"))

detok = requests.post("http://localhost:8000/detokenize", json={"model": MODEL, "tokens": tok["tokens"]}).json()
print("\n===== the exact text the model reads =====")
print(detok.get("prompt", detok))

## Step 9 · The server's side of the story
Look at the end of the server log. You'll see one line per request (`POST /v1/chat/completions … 200 OK`), and every few seconds
a **stats line**: prompt and generation throughput (tokens/s), how many requests are `Running` / `Waiting`,
**`GPU KV cache usage`** (how full the drawer is) and the **`Prefix cache hit rate`** (knowledge base §3.6).

In [ ]:
!tail -n 25 vllm_server.log

## Step 10 · Many questions at once: see batching happen
First we send 8 questions **one after another**, then the same 8 **all at once**. With batching (knowledge base §4, Part 2),
"all at once" should take not much longer than a single question. Then check the log again: you should see `Running: 8 reqs`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

questions = [f"Write a SQLite query that counts the rows of a table named table_{i}." for i in range(8)]

def ask_quietly(q):
    payload = {"model": MODEL, "messages": [{"role": "user", "content": q}], "temperature": 0, "max_tokens": 200}
    return requests.post(URL, json=payload).json()

t0 = time.time()
for q in questions:
    ask_quietly(q)
print(f"one after another: {time.time() - t0:.1f} s")

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    answers = list(pool.map(ask_quietly, questions))
print(f"all at once:       {time.time() - t0:.1f} s")

In [ ]:
!grep -E "Running: [1-9]" vllm_server.log | tail -n 5

## Step 11 · Stop the server
This frees the GPU memory. Run `nvidia-smi` again: memory used should drop back to about 0.
**When you are done, also stop the Kaggle session** (power icon, top right), because an open GPU session keeps using your weekly quota.

In [ ]:
server.terminate()
server.wait()
log_file.close()
print("server stopped")

In [ ]:
!nvidia-smi